In [3]:
%%writefile /content/pdf_web_crawler.py
#!/usr/bin/env python3
"""
pdf_web_crawler.py
==================
Crawls seed websites → discovers ALL PDFs → downloads them →
rebuilds FAISS + BM25 indexes.

Drive folders:
  FranchiseOps → MyDrive/FranchiseOps_AI/rag_pdfs/
  FreightQuote → MyDrive/FreightQuote_AI/rag_pdfs/
"""

import os
import re
import json
import time
import pickle
import hashlib
import requests
import urllib.parse
import subprocess
import sys

from pathlib import Path
from collections import deque
from datetime import datetime

# Install deps silently
for pkg in ["requests", "beautifulsoup4", "lxml", "tqdm",
            "PyMuPDF", "sentence-transformers", "faiss-cpu", "rank-bm25"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                   capture_output=True)

from bs4 import BeautifulSoup
from tqdm.auto import tqdm

# ─── Drive folder names ───────────────────────────────────────────────────────
FC_DRIVE_FOLDER = "FranchiseOps_AI"


# ─── Crawler settings ─────────────────────────────────────────────────────────
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                   "Chrome/120.0.0.0 Safari/537.36"),
    "Accept": "text/html,application/xhtml+xml,*/*;q=0.8",
}
PDF_HEADERS = {
    "User-Agent": HEADERS["User-Agent"],
    "Accept": "application/pdf,*/*",
}

MAX_CRAWL_DEPTH   = 2
MAX_PDFS_PER_SEED = 50
DOWNLOAD_TIMEOUT  = 25
REQUEST_TIMEOUT   = 15
DELAY_BETWEEN     = 0.4
CHUNK_WORDS       = 400
CHUNK_OVERLAP     = 50

# ─────────────────────────────────────────────────────────────────────────────
# SEED URLS
# ─────────────────────────────────────────────────────────────────────────────

FRANCHISE_SEEDS = [
    ("Web Source", "https://www.osha.gov/bloodborne-pathogens"),
    ("Web Source", "https://www.fda.gov/food/retail-food-protection"),
    ("Web Source", "https://www.fssai.gov.in/cms/food-safety-and-standards-act-2006.php"),
    ("Web Source", "https://www.fssai.gov.in/cms/draft-notifications.php"),
    ("Web Source", "https://www.mofpi.gov.in/"),
    ("Web Source", "https://www.servsafe.com/food-manager"),
    ("Web Source", "https://www.cdc.gov/foodsafety/index.html"),
    ("Web Source", "https://www.who.int/news-room/fact-sheets/detail/food-safety"),
    ("Web Source", "https://www.fssai.gov.in/cms/gazette-notifications.php"),
    ("Web Source", "https://www.fssai.gov.in/cms/manuals-of-methods-of-analysis-of-various-food-products.php"),
    ("Web Source", "https://labour.gov.in/maternity-benefit-act"),
    ("Web Source", "https://www.osha.gov/restaurants"),
    ("Web Source", "https://www.mygfsi.com/"),
    ("Web Source", "https://www.franchise.org/franchise-information/franchise-legal"),
    ("Web Source", "https://www.who.int/health-topics/food-safety"),
    ("Web Source", "https://www.fda.gov/food/guidance-regulation-food-and-dietary-supplements"),
    ("Web Source", "https://www.franchise.org/franchise-information/franchise-finance"),
    ("Web Source", "https://www.cdc.gov/foodsafety/outbreaks/index.html"),
    ("Web Source", "https://labour.gov.in/employees-provident-fund-organisation"),
    ("Web Source", "https://www.fda.gov/food/hazard-analysis-critical-control-point-haccp"),
    ("Web Source", "https://www.osha.gov/emergency-preparedness"),
    ("Web Source", "https://www.osha.gov/personal-protective-equipment"),
    ("Web Source", "https://www.franchise.org/franchise-information/franchise-business-opportunities"),
    ("Web Source", "https://www.franchise.org/franchise-opportunities/retail"),
    ("Web Source", "https://www.fda.gov/food/foodborne-pathogens"),
    ("Web Source", "https://labour.gov.in/social-security"),
    ("Web Source", "https://www.nraef.org/servsafe/"),
    ("Web Source", "https://www.fda.gov/food/food-safety-modernization-act-fsma"),
    ("Web Source", "https://www.fda.gov/food/food-safety-during-emergencies"),
    ("Web Source", "https://www.brcgs.com/our-standards/food-safety/"),
    ("Web Source", "https://www.tea.in/"),
    ("Web Source", "https://www.fssai.gov.in/cms/notices.php"),
    ("Web Source", "https://labour.gov.in/industrial-relations"),
    ("Web Source", "https://www.franchise.org/franchise-information/franchise-management"),
    ("Web Source", "https://labour.gov.in/employees-state-insurance-corporation"),
    ("Web Source", "https://www.franchise.org/franchise-information/franchise-marketing"),
    ("Web Source", "https://consumeraffairs.nic.in/"),
    ("Web Source", "https://dca.nic.in/"),
    ("Web Source", "https://bis.gov.in/"),
    ("Web Source", "https://ncdrc.nic.in/"),
    ("Web Source", "https://www.franchise.org/franchise-information/franchise-trends"),
    ("Web Source", "https://labour.gov.in/occupational-safety-and-health"),
    ("Web Source", "https://www.fssai.gov.in/cms/regulations.php"),
    ("Web Source", "https://www.india.gov.in/topics/industries/food-processing"),
    ("Web Source", "https://www.osha.gov/young-workers"),
    ("Web Source", "https://labour.gov.in/minimum-wages-act"),
    ("Web Source", "https://www.coffee.org.in/"),
    ("Web Source", "https://www.franchise.org/franchise-information/buying-a-franchise"),
    ("Web Source", "https://www.fssai.gov.in/cms/rules.php"),
    ("Web Source", "https://www.indianspices.com/"),
    ("Web Source", "https://www.osha.gov/fall-protection"),
    ("Web Source", "https://www.franchise.org/franchise-opportunities/food"),
    ("Web Source", "https://labour.gov.in/factories-act"),
    ("Web Source", "https://labour.gov.in/payment-of-wages-act"),
    ("Web Source", "https://mpeda.gov.in/"),
    ("Web Source", "https://www.servsafe.com/food-handler"),
    ("Web Source", "https://www.cdc.gov/foodsafety/prevention.html"),
    ("Web Source", "https://www.franchise.org/franchise-information/starting-a-franchise"),
    ("Web Source", "https://www.osha.gov/ergonomics"),
    ("Web Source", "https://www.osha.gov/retail-facilities"),
    ("Web Source", "https://labour.gov.in/child-labour"),
    ("Web Source", "https://www.iso.org/iso-22000-food-safety-management.html"),
    ("Web Source", "https://www.sqfi.com/"),
    ("Web Source", "https://www.osha.gov/heat-exposure"),
    ("Web Source", "https://www.osha.gov/hazard-communication"),
    ("Food Safety",       "https://www.who.int/publications/i/item/9789241549691"),
    ("Food Safety",       "https://www.fao.org/food-safety/resources/en/"),
    ("Food Safety",       "https://fssai.gov.in/cms/food-safety-and-standards-regulations.php"),
    ("Food Safety",       "https://fssai.gov.in/cms/resources.php"),
    ("Food Safety",       "https://www.fda.gov/food/guidance-documents-regulatory-information-topic/guidance-documents-food-labeling"),
    ("Food Standards",    "https://www.fao.org/fao-who-codexalimentarius/codex-texts/codes-of-practice/en/"),
    ("Food Standards",    "https://www.fao.org/fao-who-codexalimentarius/codex-texts/list-standards/en/"),
    ("HR Management",     "https://www.ilo.org/global/publications/lang--en/index.htm"),
    ("HR Management",     "https://www.ilo.org/wcmsp5/groups/public/---ed_emp/---emp_ent/documents/"),
    ("India Compliance",  "https://labour.gov.in/publications"),
    ("India Compliance",  "https://fssai.gov.in/upload/uploadfiles/files/"),
    ("India Compliance",  "https://mofpi.gov.in/resources/publications"),
    ("Supply Chain",      "https://www.fao.org/food-loss-and-food-waste/resources/en/"),
    ("Marketing",         "https://unctad.org/topic/ecommerce-and-digital-economy/e-commerce"),
    ("Financial",         "https://www.cbic.gov.in/resources//htdocs-cbec/gst/"),
    ("Financial",         "https://rbidocs.rbi.org.in/rdocs/Publications/"),
    ("Quality",           "https://www.unido.org/sites/default/files/2009-05/"),
    ("Sustainability",    "https://www.fao.org/sustainable-food-systems/en/"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=retail+inventory+machine+learning"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=employee+attrition+prediction"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=customer+satisfaction+NLP"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=franchise+operations+analytics"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=demand+forecasting+deep+learning+retail"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=anomaly+detection+time+series"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=RAG+retrieval+augmented+generation"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=knowledge+graph+enterprise"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=marketing+ROI+optimization"),
    ("World Bank",        "https://openknowledge.worldbank.org/search?query=franchise+india+retail"),
    ("World Bank",        "https://openknowledge.worldbank.org/search?query=food+safety+supply+chain"),
    ("World Bank",        "https://openknowledge.worldbank.org/search?query=msme+india+workforce"),
    ("OECD",              "https://www.oecd.org/industry/smes/"),
    ("OECD",              "https://www.oecd.org/consumer/"),
    ("Training",          "https://nsdcindia.org/resources/"),
    ("Training",          "https://www.msde.gov.in/en/publications"),
    ("IFC",               "https://www.ifc.org/en/insights-reports"),
    ("India Statistics",  "https://mospi.gov.in/publication"),
    ("NABARD",            "https://www.nabard.org/content.aspx?id=572"),
    ("RBI",               "https://rbidocs.rbi.org.in/rdocs/Publications/"),
    ("SIDBI",             "https://www.sidbi.in/en/home"),
    ("Codex",             "https://www.fao.org/fao-who-codexalimentarius/codex-texts/dbs/CXS/en/"),
    ("ILO",               "https://www.ilo.org/wcmsp5/groups/public/---ed_dialogue/---sector/documents/"),
    ("APEDA",             "https://apeda.gov.in/apedawebsite/six_head_product/"),
]




# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def is_pdf_url(url):
    low = url.lower()
    return (low.endswith(".pdf")
            or "/pdf/" in low
            or "download=pdf" in low
            or ".pdf?" in low
            or "format=pdf" in low)


def same_domain(url, base):
    try:
        return urllib.parse.urlparse(url).netloc == urllib.parse.urlparse(base).netloc
    except Exception:
        return False


def title_from_url(url):
    name = url.rstrip("/").split("/")[-1]
    # strip extension safely without regex escape warning
    for ext in [".pdf", ".PDF", ".htm", ".html", ".aspx", ".php"]:
        if name.endswith(ext):
            name = name[: -len(ext)]
    return name.replace("-", " ").replace("_", " ").strip()[:100]


def discover_pdfs(url, category, visited=None, found=None,
                  max_depth=MAX_CRAWL_DEPTH, max_per_seed=MAX_PDFS_PER_SEED):
    if visited is None:
        visited = set()
    if found is None:
        found = []

    # ArXiv special handling
    if "arxiv.org/search" in url:
        return _crawl_arxiv(url, category, max_per_seed)

    queue = deque([(url, 0)])
    visited.add(url)

    while queue and len(found) < max_per_seed:
        cur_url, depth = queue.popleft()
        try:
            r = requests.get(cur_url, headers=HEADERS,
                             timeout=REQUEST_TIMEOUT, allow_redirects=True)
            if r.status_code != 200:
                continue
            ctype = r.headers.get("content-type", "")

            if "pdf" in ctype.lower():
                if cur_url not in [f["url"] for f in found]:
                    found.append({"url": cur_url, "category": category,
                                  "title": title_from_url(cur_url)})
                continue

            soup = BeautifulSoup(r.text, "lxml")

            for tag in soup.find_all("a", href=True):
                href = tag["href"].strip()
                abs_href = urllib.parse.urljoin(cur_url, href)
                if is_pdf_url(abs_href) and abs_href not in [f["url"] for f in found]:
                    title = (tag.get_text(strip=True) or title_from_url(abs_href))[:120]
                    found.append({"url": abs_href, "category": category, "title": title})
                    if len(found) >= max_per_seed:
                        break

            if depth < max_depth:
                for tag in soup.find_all("a", href=True):
                    href = tag["href"].strip()
                    abs_href = urllib.parse.urljoin(cur_url, href)
                    if (abs_href not in visited
                            and same_domain(abs_href, url)
                            and not is_pdf_url(abs_href)
                            and abs_href.startswith("http")
                            and "#" not in abs_href
                            and len(visited) < 200):
                        visited.add(abs_href)
                        queue.append((abs_href, depth + 1))

        except Exception:
            pass
        time.sleep(DELAY_BETWEEN)

    return found


def _crawl_arxiv(search_url, category, limit=30):
    found = []
    try:
        r = requests.get(search_url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
        soup = BeautifulSoup(r.text, "lxml")
        for li in soup.select("li.arxiv-result")[:limit]:
            title_tag = li.select_one("p.title")
            pdf_link  = li.select_one('a[href*="/pdf/"]')
            if pdf_link:
                pdf_url = pdf_link["href"]
                if "arxiv.org" not in pdf_url:
                    pdf_url = "https://arxiv.org" + pdf_url
                if not pdf_url.endswith(".pdf"):
                    pdf_url += ".pdf"
                title = title_tag.get_text(strip=True)[:120] if title_tag else title_from_url(pdf_url)
                found.append({"url": pdf_url, "category": category, "title": title})
    except Exception:
        pass
    return found


def download_pdf(entry, out_dir):
    url   = entry["url"]
    title = entry["title"]
    # safe filename — no regex needed
    safe  = "".join(c if c.isalnum() or c in " _-" else "_"
                    for c in f"{entry['category']}___{title}")[:100]
    fname = safe + ".pdf"
    fpath = out_dir / fname

    if fpath.exists() and fpath.stat().st_size > 1024:
        return True, fpath, "cached"

    try:
        r = requests.get(url, headers=PDF_HEADERS,
                         timeout=DOWNLOAD_TIMEOUT, allow_redirects=True, stream=True)
        ctype = r.headers.get("content-type", "")
        if r.status_code == 200:
            content = b"".join(r.iter_content(8192))
            fpath.write_bytes(content)
            kind = "pdf" if "pdf" in ctype.lower() else "html"
            return True, fpath, kind
        return False, fpath, "http_" + str(r.status_code)
    except Exception as e:
        return False, fpath, "err_" + str(e)[:40]


# ─────────────────────────────────────────────────────────────────────────────
# TEXT EXTRACTION + CHUNKING
# ─────────────────────────────────────────────────────────────────────────────

def extract_text(fpath):
    try:
        import fitz
        doc  = fitz.open(str(fpath))
        text = "\n".join(p.get_text() for p in doc)
        doc.close()
        return text
    except Exception:
        pass
    try:
        raw  = fpath.read_bytes()
        soup = BeautifulSoup(raw, "lxml")
        return soup.get_text(separator="\n", strip=True)
    except Exception:
        return ""


def chunk_text(text, title, source_url,
               chunk_words=CHUNK_WORDS, overlap=CHUNK_OVERLAP):
    words  = text.split()
    step   = chunk_words - overlap
    chunks = []
    for i in range(0, max(1, len(words) - overlap), step):
        piece = " ".join(words[i: i + chunk_words])
        if len(piece) > 80:
            cid = hashlib.md5((source_url + str(i)).encode()).hexdigest()[:12]
            chunks.append({
                "chunk_id": cid,
                "text":     piece,
                "title":    title,
                "source":   source_url,
                "offset":   i,
            })
    return chunks


# ─────────────────────────────────────────────────────────────────────────────
# RAG INDEX BUILD  (matches existing RAG notebook format exactly)
# ─────────────────────────────────────────────────────────────────────────────

def build_rag_index(chunks, out_base, project_key):
    import numpy as np
    import faiss
    from sentence_transformers import SentenceTransformer
    from rank_bm25 import BM25Okapi

    faiss_dir = out_base / "faiss_indexes"
    bm25_dir  = out_base / "bm25_indexes"
    faiss_dir.mkdir(parents=True, exist_ok=True)
    bm25_dir.mkdir(parents=True, exist_ok=True)

    ST_CACHE = "/content/.cache/sentence_transformers"
    print("  Loading sentence-transformer (all-MiniLM-L6-v2)...")
    model = SentenceTransformer("all-MiniLM-L6-v2", cache_folder=ST_CACHE)

    texts = [c["text"] for c in chunks]
    print(f"  Encoding {len(texts)} chunks...")
    embs  = model.encode(texts, show_progress_bar=True,
                         batch_size=64, convert_to_numpy=True).astype("float32")

    faiss.normalize_L2(embs)
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)

    idx_path  = str(faiss_dir / (project_key + "_faiss.index"))
    meta_path = str(faiss_dir / (project_key + "_chunks_meta.json"))
    bm25_path = str(bm25_dir  / (project_key + "_bm25.pkl"))

    faiss.write_index(index, idx_path)
    with open(meta_path, "w") as f:
        json.dump(chunks, f)

    tokenized = [c["text"].lower().split() for c in chunks]
    bm25      = BM25Okapi(tokenized)
    with open(bm25_path, "wb") as f:
        pickle.dump({"bm25": bm25, "chunks": chunks}, f)

    print("  FAISS index:  " + idx_path)
    print("  Chunk meta:   " + meta_path)
    print("  BM25 index:   " + bm25_path)
    return index.ntotal


# ─────────────────────────────────────────────────────────────────────────────
# MAIN RUNNER
# ─────────────────────────────────────────────────────────────────────────────

def run_project(project_name, project_key, seeds, out_base_str):
    out_base = Path(out_base_str)
    pdf_dir  = out_base / "rag_pdfs"
    pdf_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 65)
    print("  PROJECT: " + project_name)
    print("  Seed sites: " + str(len(seeds)))
    print("  Output: " + str(out_base))
    print("=" * 65)

    # Step 1 — crawl
    print("\n  [1/4] Crawling " + str(len(seeds)) + " seed sites for PDFs...")
    all_found  = []
    visited_gl = set()
    for cat, seed_url in tqdm(seeds, desc="  Crawling"):
        found = discover_pdfs(seed_url, cat, visited=visited_gl)
        all_found.extend(found)
        print("    " + cat[:20].ljust(22) + "| " + seed_url[:50] +
              " -> " + str(len(found)) + " PDFs")

    # Deduplicate
    seen, unique = set(), []
    for e in all_found:
        if e["url"] not in seen:
            seen.add(e["url"])
            unique.append(e)
    print("\n  Unique PDFs discovered: " + str(len(unique)))

    # Step 2 — download
    print("\n  [2/4] Downloading " + str(len(unique)) + " PDFs...")
    manifest = {
        "project": project_name,
        "generated_at": datetime.now().isoformat(),
        "total_discovered": len(unique),
        "downloaded": 0, "cached": 0, "failed": 0,
        "files": [],
    }
    for entry in tqdm(unique, desc="  Downloading"):
        ok, fpath, status = download_pdf(entry, pdf_dir)
        entry["local_path"] = str(fpath)
        entry["status"]     = status
        if ok:
            if status == "cached":
                manifest["cached"] += 1
            else:
                manifest["downloaded"] += 1
        else:
            manifest["failed"] += 1
        manifest["files"].append(entry)
        time.sleep(DELAY_BETWEEN)

    with open(out_base / "manifest.json", "w") as f:
        json.dump(manifest, f, indent=2)

    print("  Downloaded: " + str(manifest["downloaded"]) +
          " | Cached: " + str(manifest["cached"]) +
          " | Failed: " + str(manifest["failed"]))

    # Step 3 — chunk
    print("\n  [3/4] Extracting text and chunking...")
    all_chunks = []
    ok_files   = [e for e in manifest["files"]
                  if e.get("status") in ("pdf", "html", "cached")]
    for entry in tqdm(ok_files, desc="  Chunking"):
        fpath = Path(entry.get("local_path", ""))
        if not fpath.exists() or fpath.stat().st_size < 512:
            continue
        text = extract_text(fpath)
        if len(text.split()) < 20:
            continue
        all_chunks.extend(chunk_text(text, entry["title"], entry["url"]))
    print("  Total chunks: " + str(len(all_chunks)))

    # Step 4 — index
    print("\n  [4/4] Building FAISS + BM25 indexes...")
    n = 0
    if all_chunks:
        n = build_rag_index(all_chunks, out_base, project_key)
    else:
        print("  No chunks to index")
    print("\n  " + project_name + " COMPLETE: " + str(n) + " vectors")
    return manifest


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    DRIVE = Path("/content/drive/MyDrive")

    if DRIVE.exists():
        FC_BASE = str(DRIVE / FC_DRIVE_FOLDER)

        print("Google Drive mounted.")
        print("FranchiseOps -> " + FC_BASE)

    else:
        FC_BASE = "/content/FranchiseOps_AI"

        print("Drive not mounted — saving locally.")

    print("FranchiseOps seed sites: " + str(len(FRANCHISE_SEEDS)))


    fc_m = run_project("FranchiseOps AI", "franchise", FRANCHISE_SEEDS, FC_BASE)


    print("\n" + "=" * 65)
    print("  FINAL SUMMARY")
    print("=" * 65)
    for nm, m in [("FranchiseOps", fc_m)]:
        print("  " + nm.ljust(14) +
              " | Found: " + str(m["total_discovered"]) +
              " | DL: " + str(m["downloaded"]) +
              " | Cached: " + str(m["cached"]) +
              " | Failed: " + str(m["failed"]))
    print("=" * 65)


Writing /content/pdf_web_crawler.py


In [4]:
!pip install -q requests beautifulsoup4 lxml tqdm PyMuPDF sentence-transformers faiss-cpu rank-bm25
!python3 /content/pdf_web_crawler.py

Drive not mounted — saving locally.
FranchiseOps seed sites: 107

  PROJECT: FranchiseOps AI
  Seed sites: 107
  Output: /content/FranchiseOps_AI

  [1/4] Crawling 107 seed sites for PDFs...
  Crawling:   0% 0/107 [00:00<?, ?it/s]    Web Source            | https://www.osha.gov/bloodborne-pathogens -> 50 PDFs
  Crawling:   1% 1/107 [00:05<10:08,  5.74s/it]    Web Source            | https://www.fda.gov/food/retail-food-protection -> 0 PDFs
  Crawling:   2% 2/107 [00:05<04:22,  2.50s/it]    Web Source            | https://www.fssai.gov.in/cms/food-safety-and-stand -> 0 PDFs
  Crawling:   3% 3/107 [00:21<15:00,  8.66s/it]    Web Source            | https://www.fssai.gov.in/cms/draft-notifications.p -> 0 PDFs
  Crawling:   4% 4/107 [00:37<19:26, 11.33s/it]    Web Source            | https://www.mofpi.gov.in/ -> 50 PDFs
  Crawling:   5% 5/107 [00:42<15:28,  9.10s/it]    Web Source            | https://www.servsafe.com/food-manager -> 0 PDFs
  Crawling:   6% 6/107 [00:42<10:18,  6.12s/it]  

In [5]:
!pip install -q faiss-cpu sentence-transformers rank-bm25 pymupdf
!pip install -q langchain langchain-community langchain-text-splitters
!pip install -q plotly pandas numpy scikit-learn joblib faker vaderSentiment
print('All dependencies installed')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 7.9 MB/s eta 0:00:00
All dependencies installed


In [6]:
from google.colab import drive
drive.mount('/content/drive')
import os

DRIVE_BASE = '/content/drive/MyDrive/FranchiseOps_AI'
FAISS_DIR  = f'{DRIVE_BASE}/faiss_indexes'
BM25_DIR   = f'{DRIVE_BASE}/bm25_indexes'
ML_DIR     = f'{DRIVE_BASE}/ml_models'
PDF_DIR    = f'{DRIVE_BASE}/rag_pdfs'
KB_DIR     = f'{DRIVE_BASE}/synthetic_kb'
DB_PATH    = f'{DRIVE_BASE}/franchise_database.db'
LLM_CACHE  = '/content/.cache/hf_models'
ST_CACHE   = '/content/.cache/sentence_transformers'

for d in [FAISS_DIR, BM25_DIR, ML_DIR, PDF_DIR, KB_DIR, LLM_CACHE, ST_CACHE]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted. Storage layout:')
print(f'  FAISS index   -> {FAISS_DIR}')
print(f'  BM25 index    -> {BM25_DIR}')
print(f'  ML models     -> {ML_DIR}')
print(f'  PDF uploads   -> {PDF_DIR}')
print(f'  Knowledge KB  -> {KB_DIR}')
print(f'  Database      -> {DB_PATH}')
print(f'  LLM cache     -> {LLM_CACHE}  (local only - too large for Drive)')

Mounted at /content/drive
Drive mounted. Storage layout:
  FAISS index   -> /content/drive/MyDrive/FranchiseOps_AI/faiss_indexes
  BM25 index    -> /content/drive/MyDrive/FranchiseOps_AI/bm25_indexes
  ML models     -> /content/drive/MyDrive/FranchiseOps_AI/ml_models
  PDF uploads   -> /content/drive/MyDrive/FranchiseOps_AI/rag_pdfs
  Knowledge KB  -> /content/drive/MyDrive/FranchiseOps_AI/synthetic_kb
  Database      -> /content/drive/MyDrive/FranchiseOps_AI/franchise_database.db
  LLM cache     -> /content/.cache/hf_models  (local only - too large for Drive)


In [7]:
import json

DOCS = [
    {'title': 'SOP-001: Customer Service Standards',
     'content': 'All franchise outlets must maintain minimum CSAT score of 4.0/5.0. Staff must greet every customer within 30 seconds. Complaint resolution must be completed within 24 hours. Monthly mystery shopping audits are conducted at all Tier 1 outlets. Customer feedback cards are collected at every transaction and digitised weekly. Any CSAT below 3.5 triggers immediate branch manager review. Digital feedback via QR codes increases response rate by 4x versus paper forms.'},
    {'title': 'SOP-002: Inventory Management Protocol',
     'content': 'Inventory reorder must trigger automatically when stock falls below 20% of monthly demand. Weekly stock audits are mandatory for Food and Beverage categories. Expired stock must be quarantined and disposed within 48 hours. FIFO (First In First Out) must be followed for all perishable items. Supplier lead time must be accounted in reorder calculations. AI-driven demand forecasting reduces wastage by 23% and stockout incidents by 41%. Optimal reorder point = Average Daily Demand x Lead Time + Safety Stock.'},
    {'title': 'SOP-003: Staff Attrition Management',
     'content': 'Staff attrition above 15% per quarter requires immediate HR intervention and root cause analysis. Exit interviews are mandatory for all departing employees. Job satisfaction surveys are administered quarterly. High performers with job_satisfaction >= 4 and tenure > 2 years are eligible for Fast Track Promotion. Overtime hours must not exceed 20 hours per month per employee. Work-life balance scores correlate 0.74 with 12-month retention. Monthly 1:1 meetings between manager and staff improve satisfaction by 18%.'},
    {'title': 'SOP-004: Marketing Campaign Compliance',
     'content': 'All marketing campaigns must be approved by Regional Marketing Manager before launch. Campaigns must achieve minimum ROI of 1.5x to be considered successful. Digital channels must follow brand guidelines including logo usage, color palettes, and tone. Social media posts require 24-hour approval window. Campaign performance is reviewed weekly. Budget allocation recommended: 40% Digital, 25% Social Media, 20% Influencer, 15% Print. Digital channels deliver highest ROI at 3.2x average.'},
    {'title': 'SOP-005: Audit and Compliance Framework',
     'content': 'Audit compliance score below 70 triggers mandatory corrective action plan within 72 hours. Hygiene audits are conducted monthly by internal QA team. Safety compliance checks occur bi-weekly. Financial audits are quarterly. Brand standards audit happens annually. Failed audits require re-audit within 30 days. Consecutive failures result in franchise review board escalation. Pre-audit self-assessment using 45-point checklist catches 78% of issues before formal review.'},
    {'title': 'SOP-006: Outlet Tier Classification',
     'content': 'Tier 1 outlets: Revenue above 150000 INR per month, CSAT >= 4.3, Staff >= 12 employees. Tier 2 outlets: Revenue 80000-150000 INR per month, CSAT >= 3.8, Staff 6-12. Tier 3 outlets: Revenue below 80000 INR per month. Classification reviewed quarterly. Tier upgrades require 3 consecutive quarters of performance above threshold. Tier 1 outlets receive priority marketing support and dedicated account manager.'},
    {'title': 'SOP-007: Digital Transformation Requirements',
     'content': 'All outlets must use the FranchiseOps AI platform for inventory tracking, staff scheduling, and sales reporting. Paper-based systems must be fully digitized by Q4. POS system integration is mandatory for all Tier 1 and Tier 2 outlets. Real-time sales data must sync to central database every 15 minutes. Outlets failing digital compliance face penalty points in annual review.'},
    {'title': 'SOP-008: Health and Safety Standards',
     'content': 'Temperature logs for refrigerated items must be recorded every 4 hours. Commercial refrigerators must maintain temperature at or below 4 degrees Celsius. All food handlers must possess valid food safety certification renewed annually. Emergency evacuation procedures must be posted and drilled quarterly. First aid kits must be fully stocked and inspected monthly. CCTV systems must have 30-day footage retention minimum.'},
    {'title': 'SOP-009: Financial Management and Reporting',
     'content': 'Daily revenue must be reconciled and submitted to Regional Finance by 11 PM. Cash variance greater than 2% triggers immediate investigation. P&L reports are submitted monthly on the 5th business day. Operating cost ratio must not exceed 65% of revenue. Industry benchmark operating cost ratio is 58-62%. Labor cost should not exceed 28% of revenue. Capital expenditure above 50000 INR requires Franchise Owner approval.'},
    {'title': 'SOP-010: Crisis Management Protocol',
     'content': 'Food safety incidents must be reported to Regional Manager within 1 hour. Customer injury claims must be escalated to legal team immediately. Media inquiries must be routed to Corporate Communications. Outlet closure decisions rest with Regional Director. Post-crisis report must be submitted within 7 days. All incidents are logged in the FranchiseOps AI alert system with resolution tracking.'},
    {'title': 'BP-001: Revenue Growth Strategies',
     'content': 'Top-performing franchise outlets achieve revenue growth through three pillars: upselling training increases average transaction value by 18%, loyalty program enrollment means customers spend 2.3x more, digital ordering channels reduce wait time by 40% and increase throughput by 25%. Weekend promotions consistently yield 30% higher revenue versus weekday baselines. Festival season October to January typically 40% above monthly average.'},
    {'title': 'BP-002: Staff Retention Playbook',
     'content': 'Outlets with lowest attrition share these practices: structured onboarding in first 30 days, monthly 1-on-1 meetings between manager and each staff member, peer recognition programs, clear promotion pathways at 6 months and 18 months, and competitive compensation reviewed annually. Outlets with AI-powered scheduling save 12% on labor costs versus manual scheduling.'},
    {'title': 'BP-003: Inventory Optimization Techniques',
     'content': 'AI-powered demand forecasting reduces wastage by 23% and stockout incidents by 41%. Vendor consolidation to 3-5 primary suppliers reduces procurement costs by 12%. Safety stock should be 15% of weekly demand for fast movers and 30% for seasonal items. FIFO is critical for perishables. Real-time inventory visibility prevents phantom stockouts and over-ordering.'},
    {'title': 'BP-004: Customer Experience Excellence',
     'content': 'CSAT above 4.5 is associated with 67% higher likelihood of repeat visits. Key drivers of positive CSAT: speed of service accounts for 35% of score, product quality 30%, staff attitude 25%, cleanliness 10%. Mystery shopping scores below 75 require immediate manager coaching. NPS target is 45 plus versus industry average of 32.'},
    {'title': 'BP-005: Marketing ROI Maximization',
     'content': 'Digital channels deliver highest ROI at 3.2x average. Email marketing to loyalty database achieves 28% open rate versus 18% industry average. Geo-targeted promotions within 3km radius convert at 2.1x versus broad campaigns. Budget allocation: 40% Digital, 25% Social, 20% Influencer, 15% Print. Influencer marketing with micro-influencers under 50000 followers delivers 7x higher engagement than celebrity endorsements.'},
    {'title': 'KPI-001: Revenue Benchmarks',
     'content': 'National average revenue per outlet is 95000 INR per month. Top quartile threshold is 140000 INR per month. Bottom quartile threshold is 55000 INR per month. Mumbai Delhi Bangalore outlets average 35% higher revenue than Tier 2 cities. Peak revenue months October to January during festival season typically 40% above monthly average. Lowest revenue June to August monsoon season.'},
    {'title': 'KPI-002: Operational Efficiency Benchmarks',
     'content': 'Industry benchmark operating cost ratio is 58-62% of revenue. Best-in-class outlets achieve 48-52%. Labor cost should not exceed 28% of revenue. Food cost benchmark is 25-30% of F&B revenue. Energy costs 5-8% of revenue. Franchise fee typically 6-8% of gross revenue. Outlets with AI-powered scheduling save 12% on labor costs versus manual scheduling.'},
    {'title': 'KPI-003: Customer Metrics Benchmarks',
     'content': 'Industry average CSAT is 3.6 out of 5.0. FranchiseOps network average is 4.1. Target CSAT for Tier 1 is 4.3 or above and for Tier 2 is 4.0 or above. Net Promoter Score target is 45 versus industry average of 32. Customer return rate within 30 days target is 65% or above. Average transaction value target is 320 INR versus industry average of 260 INR.'},
    {'title': 'ML-001: Attrition Prediction Model',
     'content': 'The FranchiseOps AI attrition model uses Random Forest with features salary, overtime hours, job satisfaction, age, tenure years, and work-life balance. Model achieves 82% accuracy on validation set. Feature importance: job satisfaction 28%, overtime hours 22%, salary 18%, work-life balance 16%, tenure 10%, age 6%. High risk threshold is predicted probability above 0.6. Model is retrained monthly with new data.'},
    {'title': 'ML-002: Revenue Prediction and Demand Forecasting',
     'content': 'Revenue prediction uses Gradient Boosting Regressor with features operating costs, customer satisfaction, staff headcount, tier, and location. Model achieves R2 of 0.79 on validation. Demand forecasting uses time series decomposition with seasonal adjustment for festival periods. Forecast horizon is 4 weeks with weekly retraining. Inventory replenishment orders are automatically triggered based on forecast plus safety stock.'},
]

doc_path = f'{KB_DIR}/franchise_kb.jsonl'
with open(doc_path, 'w') as f:
    for doc in DOCS:
        f.write(json.dumps(doc) + '\n')

print(f'Generated {len(DOCS)} franchise knowledge documents -> Drive')
for d in DOCS:
    print(f'  * {d["title"]}')


Generated 20 franchise knowledge documents -> Drive
  * SOP-001: Customer Service Standards
  * SOP-002: Inventory Management Protocol
  * SOP-003: Staff Attrition Management
  * SOP-004: Marketing Campaign Compliance
  * SOP-005: Audit and Compliance Framework
  * SOP-006: Outlet Tier Classification
  * SOP-007: Digital Transformation Requirements
  * SOP-008: Health and Safety Standards
  * SOP-009: Financial Management and Reporting
  * SOP-010: Crisis Management Protocol
  * BP-001: Revenue Growth Strategies
  * BP-002: Staff Retention Playbook
  * BP-003: Inventory Optimization Techniques
  * BP-004: Customer Experience Excellence
  * BP-005: Marketing ROI Maximization
  * KPI-001: Revenue Benchmarks
  * KPI-002: Operational Efficiency Benchmarks
  * KPI-003: Customer Metrics Benchmarks
  * ML-001: Attrition Prediction Model
  * ML-002: Revenue Prediction and Demand Forecasting


In [8]:
import os, json

pdf_docs = []
# Scan Drive PDF folder first
for fname in os.listdir(PDF_DIR):
    if fname.endswith('.pdf'):
        try:
            import fitz
            doc = fitz.open(f'{PDF_DIR}/{fname}')
            for page_num, page in enumerate(doc):
                text = page.get_text().strip()
                if len(text) > 100:
                    pdf_docs.append({'title': f'{fname} p{page_num+1}', 'content': text})
            print(f'  Ingested: {fname}')
        except Exception as e:
            print(f'  Error: {fname}: {e}')

# Also scan /content/ for any uploaded PDFs
for fname in os.listdir('/content'):
    if fname.endswith('.pdf'):
        try:
            import fitz
            doc = fitz.open(f'/content/{fname}')
            for page_num, page in enumerate(doc):
                text = page.get_text().strip()
                if len(text) > 100:
                    pdf_docs.append({'title': f'{fname} p{page_num+1}', 'content': text})
            print(f'  Ingested /content/{fname}')
        except Exception as e:
            print(f'  Error: {fname}: {e}')

if pdf_docs:
    with open(f'{KB_DIR}/pdf_extracted.jsonl', 'w') as f:
        for d in pdf_docs:
            f.write(json.dumps(d) + '\n')
    print(f'Saved {len(pdf_docs)} PDF pages -> Drive')
else:
    print('No PDFs found. Upload to Drive at: ' + PDF_DIR)
    print('Using synthetic KB only - fully functional for demo.')


No PDFs found. Upload to Drive at: /content/drive/MyDrive/FranchiseOps_AI/rag_pdfs
Using synthetic KB only - fully functional for demo.


In [9]:
import json, os

def chunk_text(text, size=400, overlap=80):
    words = text.split()
    chunks, i = [], 0
    while i < len(words):
        chunks.append(' '.join(words[i:i+size]))
        i += size - overlap
    return chunks

all_chunks, cid = [], 0

# Load all KB files from Drive
for fname in ['franchise_kb.jsonl', 'pdf_extracted.jsonl']:
    fpath = f'{KB_DIR}/{fname}'
    if not os.path.exists(fpath):
        continue
    with open(fpath) as f:
        for line in f:
            doc = json.loads(line)
            for chunk in chunk_text(doc['content']):
                all_chunks.append({
                    'chunk_id': f'fops_{cid:05d}',
                    'title': doc['title'],
                    'text': chunk,
                    'source': fname.replace('.jsonl',''),
                    'words': len(chunk.split())
                })
                cid += 1

chunks_path = f'{KB_DIR}/all_chunks.jsonl'
with open(chunks_path, 'w') as f:
    for c in all_chunks:
        f.write(json.dumps(c) + '\n')

print(f'Chunking complete:')
print(f'  Total chunks: {len(all_chunks)}')
print(f'  Avg chunk size: {sum(c["words"] for c in all_chunks)/max(1,len(all_chunks)):.0f} words')
print(f'  Saved to Drive: {chunks_path}')


Chunking complete:
  Total chunks: 20
  Avg chunk size: 61 words
  Saved to Drive: /content/drive/MyDrive/FranchiseOps_AI/synthetic_kb/all_chunks.jsonl


In [10]:
import json, numpy as np, faiss, os, time
from sentence_transformers import SentenceTransformer

print('Loading sentence-transformer (cached locally)...')
t0 = time.time()
embedder = SentenceTransformer('all-MiniLM-L6-v2', cache_folder=ST_CACHE)
print(f'Model ready in {time.time()-t0:.1f}s')

with open(f'{KB_DIR}/all_chunks.jsonl') as f:
    chunks = [json.loads(l) for l in f]

texts = [c['text'] for c in chunks]
print(f'Encoding {len(texts)} chunks...')
t1 = time.time()
embeddings = embedder.encode(texts, batch_size=128, show_progress_bar=True, convert_to_numpy=True).astype('float32')
faiss.normalize_L2(embeddings)   # Cosine similarity via dot product
print(f'Encoding done in {time.time()-t1:.1f}s')

dim = embeddings.shape[1]
nlist = min(32, len(chunks) // 4)
quantizer = faiss.IndexFlatIP(dim)
index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)
index.train(embeddings)
index.add(embeddings)
index.nprobe = 8   # Search 8 clusters - good speed/recall tradeoff

idx_path  = f'{FAISS_DIR}/franchise_faiss.index'
meta_path = f'{FAISS_DIR}/franchise_chunks_meta.json'
faiss.write_index(index, idx_path)
with open(meta_path, 'w') as f:
    json.dump(chunks, f)

print(f'FAISS index saved to Drive:')
print(f'  {idx_path} ({os.path.getsize(idx_path)/1024:.1f} KB)')
print(f'  Vectors: {index.ntotal} | Dim: {dim} | Type: IVFFlat cosine')


Loading sentence-transformer (cached locally)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model ready in 11.8s
Encoding 20 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Encoding done in 1.2s
FAISS index saved to Drive:
  /content/drive/MyDrive/FranchiseOps_AI/faiss_indexes/franchise_faiss.index (37.8 KB)
  Vectors: 20 | Dim: 384 | Type: IVFFlat cosine


In [11]:
import json, pickle, os
from rank_bm25 import BM25Okapi

with open(f'{KB_DIR}/all_chunks.jsonl') as f:
    chunks = [json.loads(l) for l in f]

tokenized = [c['text'].lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized)

bm25_path = f'{BM25_DIR}/franchise_bm25.pkl'
with open(bm25_path, 'wb') as f:
    pickle.dump({'bm25': bm25, 'chunks': chunks}, f)

print(f'BM25 index saved to Drive:')
print(f'  {bm25_path} ({os.path.getsize(bm25_path)/1024:.1f} KB)')
print(f'  Documents indexed: {len(chunks)}')

BM25 index saved to Drive:
  /content/drive/MyDrive/FranchiseOps_AI/bm25_indexes/franchise_bm25.pkl (29.6 KB)
  Documents indexed: 20


In [12]:
import json, numpy as np, faiss, pickle, time
from sentence_transformers import SentenceTransformer

class FranchiseRAG:
    def __init__(self):
        t0 = time.time()
        self.index    = faiss.read_index(f'{FAISS_DIR}/franchise_faiss.index')
        self.index.nprobe = 8
        with open(f'{FAISS_DIR}/franchise_chunks_meta.json') as f:
            self.chunks = json.load(f)
        with open(f'{BM25_DIR}/franchise_bm25.pkl', 'rb') as f:
            data = pickle.load(f)
        self.bm25     = data['bm25']
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2', cache_folder=ST_CACHE)
        print(f'FranchiseRAG ready in {time.time()-t0:.1f}s | {len(self.chunks)} chunks')

    def search(self, query, k=4, alpha=0.6):
        q_emb = self.embedder.encode([query], convert_to_numpy=True).astype('float32')
        faiss.normalize_L2(q_emb)
        scores, idxs = self.index.search(q_emb, k*2)
        dense = [{'chunk': self.chunks[i], 'ds': float(s)} for s, i in zip(scores[0], idxs[0]) if i >= 0]
        tokens = query.lower().split()
        bm25_s = self.bm25.get_scores(tokens)
        top_k  = np.argsort(bm25_s)[::-1][:k*2]
        sparse = [{'chunk': self.chunks[i], 'bs': float(bm25_s[i])} for i in top_k if bm25_s[i] > 0]
        scored = {}
        if dense:
            dm = max(c['ds'] for c in dense) or 1.0
            for c in dense:
                cid = c['chunk']['chunk_id']
                scored[cid] = scored.get(cid, {'chunk': c['chunk'], 'score': 0.0})
                scored[cid]['score'] += alpha * (c['ds'] / dm)
        if sparse:
            bm = max(c['bs'] for c in sparse) or 1.0
            for c in sparse:
                cid = c['chunk']['chunk_id']
                scored[cid] = scored.get(cid, {'chunk': c['chunk'], 'score': 0.0})
                scored[cid]['score'] += (1-alpha) * (c['bs'] / bm)
        ranked = sorted(scored.values(), key=lambda x: x['score'], reverse=True)[:k]
        return [{'text': r['chunk']['text'], 'title': r['chunk']['title'],
                 'source': r['chunk']['source'], 'score': r['score']} for r in ranked]

rag = FranchiseRAG()

TEST_QUERIES = [
    'What is the minimum CSAT score required?',
    'How should I handle high staff attrition?',
    'What are the Tier 1 outlet requirements?',
    'What corrective action is needed for low audit scores?',
    'How should inventory reorder be triggered?',
    'What is the recommended marketing budget allocation?',
    'What are the financial reporting requirements?',
]

print('Query | Score | Source | Latency')
print('-' * 80)
latencies = []
for q in TEST_QUERIES:
    t = time.time()
    results = rag.search(q, k=1)
    ms = (time.time()-t)*1000
    latencies.append(ms)
    r = results[0] if results else {}
    print(f'{q[:40]:42} | {r.get("score",0):.3f} | {r.get("title","")[:20]:22} | {ms:.0f}ms')
print(f'Average latency: {sum(latencies)/len(latencies):.0f}ms')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FranchiseRAG ready in 2.3s | 20 chunks
Query | Score | Source | Latency
--------------------------------------------------------------------------------
What is the minimum CSAT score required?   | 0.967 | SOP-001: Customer Se   | 106ms
How should I handle high staff attrition   | 1.000 | SOP-003: Staff Attri   | 61ms
What are the Tier 1 outlet requirements?   | 0.767 | SOP-007: Digital Tra   | 69ms
What corrective action is needed for low   | 1.000 | SOP-005: Audit and C   | 96ms
How should inventory reorder be triggere   | 1.000 | SOP-002: Inventory M   | 80ms
What is the recommended marketing budget   | 0.971 | SOP-004: Marketing C   | 83ms
What are the financial reporting require   | 0.933 | SOP-005: Audit and C   | 96ms
Average latency: 84ms


In [13]:
RAG_ENGINE = '''
import os, json, pickle, numpy as np

_retriever = None
FAISS_DIR = '/content/drive/MyDrive/FranchiseOps_AI/faiss_indexes'
BM25_DIR  = '/content/drive/MyDrive/FranchiseOps_AI/bm25_indexes'
ST_CACHE  = '/content/.cache/sentence_transformers'

def _load_retriever():
    global _retriever
    if _retriever is not None: return _retriever
    try:
        import faiss
        from sentence_transformers import SentenceTransformer
        idx = faiss.read_index(f'{FAISS_DIR}/franchise_faiss.index')
        idx.nprobe = 8
        with open(f'{FAISS_DIR}/franchise_chunks_meta.json') as f: chunks = json.load(f)
        with open(f'{BM25_DIR}/franchise_bm25.pkl', 'rb') as f: data = pickle.load(f)
        emb = SentenceTransformer('all-MiniLM-L6-v2', cache_folder=ST_CACHE)
        _retriever = {'index': idx, 'chunks': chunks, 'bm25': data['bm25'], 'embedder': emb}
        return _retriever
    except Exception as e:
        return None

def is_rag_ready():
    return os.path.exists(f'{FAISS_DIR}/franchise_faiss.index')

def retrieve(query, k=4, alpha=0.6):
    r = _load_retriever()
    if r is None:
        return [{'text': 'Run FranchiseOps_RAG_Builder.ipynb to build the index.', 'title': 'System', 'source': 'system', 'score': 0}]
    import faiss
    q_emb = r['embedder'].encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(q_emb)
    scores, idxs = r['index'].search(q_emb, k*2)
    dense = [{'chunk': r['chunks'][i], 'ds': float(s)} for s, i in zip(scores[0], idxs[0]) if i >= 0]
    tokens = query.lower().split()
    bm25_s = r['bm25'].get_scores(tokens)
    top_k  = np.argsort(bm25_s)[::-1][:k*2]
    sparse = [{'chunk': r['chunks'][i], 'bs': float(bm25_s[i])} for i in top_k if bm25_s[i] > 0]
    scored = {}
    if dense:
        dm = max(c['ds'] for c in dense) or 1.0
        for c in dense:
            cid = c['chunk']['chunk_id']
            scored[cid] = scored.get(cid, {'chunk': c['chunk'], 'score': 0.0})
            scored[cid]['score'] += alpha * (c['ds'] / dm)
    if sparse:
        bm = max(c['bs'] for c in sparse) or 1.0
        for c in sparse:
            cid = c['chunk']['chunk_id']
            scored[cid] = scored.get(cid, {'chunk': c['chunk'], 'score': 0.0})
            scored[cid]['score'] += (1-alpha) * (c['bs'] / bm)
    ranked = sorted(scored.values(), key=lambda x: x['score'], reverse=True)[:k]
    return [{'text': r['chunk']['text'], 'title': r['chunk']['title'], 'source': r['chunk']['source'], 'score': r['score']} for r in ranked]

def answer_with_citation(query):
    results = retrieve(query)
    if not results or results[0]['score'] == 0:
        return 'No relevant context found.', 'None'
    return ' '.join([r['text'] for r in results]), results[0]['title']
'''

with open('/content/rag_engine.py', 'w') as f:
    f.write(RAG_ENGINE)
print('Exported: /content/rag_engine.py')
print('This module auto-loads FAISS index from Drive on first query.')


Exported: /content/rag_engine.py
This module auto-loads FAISS index from Drive on first query.


In [14]:
import plotly.express as px, pandas as pd, os

QUERIES = [
    'What is the minimum CSAT score required?',
    'How should I handle high staff attrition?',
    'What are the Tier 1 outlet requirements?',
    'What corrective action is needed for audit scores below 70?',
    'How should inventory reorder be triggered?',
    'What is the recommended marketing budget allocation?',
    'What happens when operating cost exceeds 65%?',
    'What are the health and safety standards?',
]
all_results = []
for q in QUERIES:
    for r in rag.search(q, k=3):
        all_results.append({'Query': q[:35]+'..', 'Title': r['title'][:28], 'Score': r['score'], 'Source': r['source']})

df = pd.DataFrame(all_results)
fig = px.bar(df.groupby('Source')['Score'].mean().reset_index(),
    x='Source', y='Score', color='Source', title='Average Retrieval Score by Source')
fig.show()

fig2 = px.box(df, x='Source', y='Score', color='Source', title='Score Distribution by Source')
fig2.show()

print('=' * 60)
print('  FRANCHISEOPS RAG BUILDER - COMPLETE')
print('=' * 60)
files = [
    (f'{FAISS_DIR}/franchise_faiss.index', 'FAISS dense index'),
    (f'{FAISS_DIR}/franchise_chunks_meta.json', 'Chunk metadata'),
    (f'{BM25_DIR}/franchise_bm25.pkl', 'BM25 sparse index'),
    (f'{KB_DIR}/franchise_kb.jsonl', 'Synthetic knowledge base'),
    ('/content/rag_engine.py', 'RAG engine module'),
]
for path, desc in files:
    exists = os.path.exists(path)
    size = os.path.getsize(path)/1024 if exists else 0
    print(f'  {chr(9989) if exists else chr(10060)} {desc}: {path.split("/")[-1]} ({size:.1f} KB)')
print()
print('Files in Drive persist between Colab sessions!')
print('LLM weights stay local (/content/) to save Drive space.')
print(f'Avg retrieval score: {df["Score"].mean():.3f}')


  FRANCHISEOPS RAG BUILDER - COMPLETE
  ✅ FAISS dense index: franchise_faiss.index (37.8 KB)
  ✅ Chunk metadata: franchise_chunks_meta.json (10.6 KB)
  ✅ BM25 sparse index: franchise_bm25.pkl (29.6 KB)
  ✅ Synthetic knowledge base: franchise_kb.jsonl (9.4 KB)
  ✅ RAG engine module: rag_engine.py (2.7 KB)

Files in Drive persist between Colab sessions!
LLM weights stay local (/content/) to save Drive space.
Avg retrieval score: 0.756
